# 📧 Spam Email Detector
A deep learning classifier that detects spam SMS messages using an LSTM neural network.

**Run all cells in order:** Runtime → Run all (`Ctrl+F9`)

## Step 1 — Install & Import Libraries

In [ ]:
!pip install tensorflow pandas scikit-learn seaborn gradio --quiet

import pandas as pd
import numpy as np
import tensorflow as tf
import seaborn as sns
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Embedding, LSTM, Dropout, GlobalAveragePooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

print('✅ All libraries imported successfully!')
print(f'   TensorFlow version: {tf.__version__}')

## Step 2 — Load the Dataset

In [ ]:
url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

print('📊 Dataset Overview')
print('=' * 40)
print(df.head())
print(f'\nTotal messages : {len(df)}')
print(f'Spam           : {(df["label"] == "spam").sum()} ({(df["label"] == "spam").mean():.1%})')
print(f'Ham            : {(df["label"] == "ham").sum()} ({(df["label"] == "ham").mean():.1%})')

## Step 3 — Preprocess the Text

In [ ]:
# Config
MAX_WORDS  = 10000
MAX_LENGTH = 100

# Encode labels: spam=1, ham=0
df['label_enc'] = (df['label'] == 'spam').astype(int)

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    df['message'], df['label_enc'],
    test_size=0.2, random_state=42, stratify=df['label_enc']
)

# Tokenize
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train)

# Convert text → padded sequences
X_train_pad = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LENGTH, padding='post')
X_test_pad  = pad_sequences(tokenizer.texts_to_sequences(X_test),  maxlen=MAX_LENGTH, padding='post')

# Class weights to handle imbalance (87% ham vs 13% spam)
weights      = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_train)
class_weights = {0: weights[0], 1: weights[1]}

print('✅ Preprocessing complete!')
print(f'   Train samples : {len(X_train_pad)}')
print(f'   Test samples  : {len(X_test_pad)}')
print(f'   Class weights : Ham={class_weights[0]:.2f}, Spam={class_weights[1]:.2f}')

## Step 4 — Build the Model (LSTM)

In [ ]:
model = Sequential([
    Embedding(MAX_WORDS, 64),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1,  activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Step 5 — Train the Model

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad, y_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    class_weight=class_weights,
    callbacks=[early_stop],
    verbose=1
)

print(f'\n✅ Training stopped at epoch {len(history.history["loss"])}')

## Step 6 — Evaluate the Model

In [ ]:
# Test accuracy
loss, acc = model.evaluate(X_test_pad, y_test, verbose=0)
print(f'Test Accuracy : {acc:.4f}')
print(f'Test Loss     : {loss:.4f}\n')

# Classification report
y_pred = (model.predict(X_test_pad, verbose=0) > 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=['Ham', 'Spam']))

# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'],     label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Accuracy over epochs')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Loss over epochs')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Ham', 'Spam'],
            yticklabels=['Ham', 'Spam'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

## Step 7 — Save the Model

In [ ]:
model.save('spam_detector.keras')
print('✅ Model saved as spam_detector.keras')

# To reload it later:
# model = load_model('spam_detector.keras')

## Step 8 — Test with Your Own Messages

In [ ]:
def predict_spam(text):
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post')
    prob = model.predict(pad, verbose=0)[0][0]
    label = 'SPAM' if prob > 0.5 else 'HAM'
    print(f'Message : {text[:70]}')
    print(f'Result  : {label}  (confidence: {prob:.2%})\n')

predict_spam("Congratulations! You've won a FREE iPhone. Click here now!")
predict_spam("Hey, are we still meeting for lunch tomorrow?")
predict_spam("URGENT: Your account will be suspended. Verify NOW: bit.ly/x")
predict_spam("Can you send me the lecture slides from today?")
predict_spam("You have been selected for a cash prize of $1000. Call us now!")

## Step 9 — Interactive Web App (Gradio)

In [ ]:
import gradio as gr

def check_spam(text):
    if not text.strip():
        return 'Please enter a message.'
    seq = tokenizer.texts_to_sequences([text])
    pad = pad_sequences(seq, maxlen=MAX_LENGTH, padding='post')
    prob = model.predict(pad, verbose=0)[0][0]
    label = 'SPAM' if prob > 0.5 else 'HAM'
    return f'{label} — confidence: {prob:.2%}'

gr.Interface(
    fn=check_spam,
    inputs=gr.Textbox(lines=3, placeholder='Type a message to check...', label='Message'),
    outputs=gr.Textbox(label='Result'),
    title='Spam Detector',
    description='Enter any SMS or email text to check if it is spam.',
    examples=[
        ['Congratulations! You won a FREE prize. Call now!'],
        ['Are you coming to class tomorrow?'],
    ]
).launch()